# B.1 Grundspecifikation och årseffekt

Sekventiell uppbyggnad av Poisson-GLM (M0–M3) med jämförelse via AIC och valideringsdeviance på 2024. Träning: 2021–2023, validering: 2024, test 2025 reserverad.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

data_dir = Path("../../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")

print(f"Träning: {df.shape[0]:,} rader, {df.shape[1]} kolumner")

Träning: 1,033,386 rader, 8 kolumner


In [2]:
# Datakontroll och förberedelse
assert (df["Duration"] > 0).all(), "Duration innehåller nollor — hantera innan log"

df["Ar"] = df["Ar"].astype(int)
df["log_duration"] = np.log(df["Duration"])
df["log_omsattning"] = np.log(df["Omsattning"])

# Temporal split: träna 2021-2023, validera 2024
df_train = df[df["Ar"].isin([2021, 2022, 2023])].copy()
df_val = df[df["Ar"] == 2024].copy()

print(f"Träning (2021-2023): {df_train.shape[0]:,} rader")
print(f"Validering (2024):   {df_val.shape[0]:,} rader")

Träning (2021-2023): 755,691 rader
Validering (2024):   277,695 rader


## Modellspecifikation

Poisson-familj, log-länk, `log(Duration)` som offset. Referenskategorier: Byggföretag och Landsbyggd.

| Modell | Specifikation |
|--------|--------------|
| M0 | Intercept only + offset |
| M1 | C(Verksamhet) + C(GeografisktOmrade) + offset |
| M2 | M1 + log(Omsättning) + offset |
| M3 | M2 + C(Ar) + offset (enbart in-sample) |

In [3]:
# M0: Intercept only (null-baseline)
m0 = smf.glm(
    "AntalSkador ~ 1",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M0  AIC: {m0.aic:,.2f}   Deviance: {m0.deviance:,.2f}")

M0  AIC: 141,086.49   Deviance: 112,768.79


In [4]:
# M1: Kategoriska huvudeffekter
m1 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd'))",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M1  AIC: {m1.aic:,.2f}   Deviance: {m1.deviance:,.2f}")

M1  AIC: 139,723.38   Deviance: 111,387.69


In [5]:
# M2: Primär slutmodell — lägger till log(Omsättning)
m2 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M2  AIC: {m2.aic:,.2f}   Deviance: {m2.deviance:,.2f}")

M2  AIC: 136,971.03   Deviance: 108,633.33


In [6]:
# M3: Årseffekt (enbart in-sample — kan inte prediktera osedda år)
m3 = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning "
    "+ C(Ar)",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M3  AIC: {m3.aic:,.2f}   Deviance: {m3.deviance:,.2f}")

M3  AIC: 136,968.88   Deviance: 108,627.19


## 1. AIC-jämförelse M0–M3

In [7]:
aic_table = pd.DataFrame({
    "Modell": ["M0", "M1", "M2", "M3"],
    "Beskrivning": [
        "Intercept only",
        "Verksamhet + Geografi",
        "M1 + log(Omsättning)",
        "M2 + År",
    ],
    "Antal parametrar": [
        m0.df_model + 1,
        m1.df_model + 1,
        m2.df_model + 1,
        m3.df_model + 1,
    ],
    "AIC": [m0.aic, m1.aic, m2.aic, m3.aic],
    "Deviance": [m0.deviance, m1.deviance, m2.deviance, m3.deviance],
})

display(aic_table.style.format({"AIC": "{:,.0f}", "Deviance": "{:,.0f}"}))

,Modell,Beskrivning,Antal parametrar,AIC,Deviance
0,M0,Intercept only,1,"141,086","112,769"
1,M1,Verksamhet + Geografi,10,"139,723","111,388"
2,M2,M1 + log(Omsättning),11,"136,971","108,633"
3,M3,M2 + År,13,"136,969","108,627"


M0→M1: Stort AIC-fall (verksamhet + geografi fångar tydliga skillnader). M1→M2: Ytterligare förbättring via log(Omsättning). M2→M3: Marginell förbättring. Årseffekt liten.

## 2. Valideringsdeviance M0–M2

M3 utesluts — `C(Ar)` kan inte prediktera osedda år.

In [8]:
import warnings


def poisson_deviance(y_true, y_pred):
    """Beräkna Poisson-deviance mellan observerade och predikterade värden."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 1e-12, None)

    # Beräkna bidrag separat för nollor och icke-nollor
    mask = y_true > 0
    dev_nonzero = np.sum(y_true[mask] * np.log(y_true[mask] / y_pred[mask]))
    dev = 2 * (dev_nonzero - np.sum(y_true - y_pred))
    return dev


# Prediktera på valideringsdata (2024)
y_val = df_val["AntalSkador"].values
val_offset = df_val["log_duration"]

val_pred_m0 = m0.predict(df_val, offset=val_offset)
val_pred_m1 = m1.predict(df_val, offset=val_offset)
val_pred_m2 = m2.predict(df_val, offset=val_offset)

val_dev_table = pd.DataFrame({
    "Modell": ["M0", "M1", "M2"],
    "Beskrivning": [
        "Intercept only",
        "Verksamhet + Geografi",
        "M1 + log(Omsättning)",
    ],
    "Valideringsdeviance (2024)": [
        poisson_deviance(y_val, val_pred_m0.values),
        poisson_deviance(y_val, val_pred_m1.values),
        poisson_deviance(y_val, val_pred_m2.values),
    ],
    "Totalt predikterat": [
        val_pred_m0.sum(),
        val_pred_m1.sum(),
        val_pred_m2.sum(),
    ],
    "Totalt observerat": [
        y_val.sum(),
        y_val.sum(),
        y_val.sum(),
    ],
})

display(val_dev_table.style.format({
    "Valideringsdeviance (2024)": "{:,.0f}",
    "Totalt predikterat": "{:,.0f}",
    "Totalt observerat": "{:,.0f}",
}))

,Modell,Beskrivning,Valideringsdeviance (2024),Totalt predikterat,Totalt observerat
0,M0,Intercept only,"42,643","5,250","5,446"
1,M1,Verksamhet + Geografi,"42,105","5,247","5,446"
2,M2,M1 + log(Omsättning),"41,002","5,250","5,446"


M2 har lägst valideringsdeviance. Förbättringen från log(Omsättning) generaliserar till osedd data.

## 3. M2-koefficienter och rate ratios

Rate ratio = exp(β). Multiplikativ effekt på skadefrekvens relativt referenskategorin.

In [9]:
conf = m2.conf_int()

coef_table = pd.DataFrame({
    "Koefficient (β)": m2.params,
    "Std.fel": m2.bse,
    "Rate ratio": np.exp(m2.params),
    "KI nedre (2.5%)": np.exp(conf.iloc[:, 0]),
    "KI övre (97.5%)": np.exp(conf.iloc[:, 1]),
})

display(coef_table.round(4))

,Koefficient (β),Std.fel,Rate ratio,KI nedre (2.5%),KI övre (97.5%)
Intercept,-11.0790,0.1401,0.0000,0.0000,0.0000
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Elektriker]",-0.3713,0.0339,0.6898,0.6455,0.7372
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Grävning & Schaktning]",-0.1543,0.0339,0.8570,0.8018,0.9159
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Målare]",-0.4448,0.0349,0.6409,0.5986,0.6863
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Takarbeten]",0.1029,0.0373,1.1084,1.0301,1.1926
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.VVS]",0.3557,0.0254,1.4272,1.3579,1.5000
"C(Verksamhet, Treatment(reference='Byggföretag'))[T.Övriga specialistföretag]",-0.0193,0.0240,0.9809,0.9358,1.0281
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Mellanstorstad]",0.1858,0.0327,1.2041,1.1294,1.2838
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Småstad]",-0.2788,0.0373,0.7567,0.7034,0.8141
"C(GeografisktOmrade, Treatment(reference='Landsbyggd'))[T.Storstad]",0.3796,0.0313,1.4618,1.3748,1.5543


Rangordningen bekräftar A3: VVS högst, Målare lägst. Storstad högst geografi. log(Omsättning) positivt samband. Detaljerad tolkning i B2.

## 4. Årseffekt (M3, enbart in-sample)

Kategorisk `C(Ar)` kan inte prediktera 2024/2025. Syftet är bara att undersöka om frekvensen varierat mellan åren.

## 4b. Alternativa ekonomiska variabler

Test av Självrisk och Försäkringsbelopp som prediktiva tillägg ovanpå M2, med samma temporala validering (träning 2021–2023, validering 2024).

In [10]:
# Förberedelse: log-transformera självrisk och försäkringsbelopp
assert (df_train["Sjalvrisk"] > 0).all(), "Sjalvrisk måste vara positiv för log-transformering"
assert (df_val["Sjalvrisk"] > 0).all(), "Sjalvrisk måste vara positiv för log-transformering"
assert (df_train["Forsakringsbelopp"] > 0).all(), "Forsakringsbelopp måste vara positivt för log-transformering"
assert (df_val["Forsakringsbelopp"] > 0).all(), "Forsakringsbelopp måste vara positivt för log-transformering"

df_train["log_sjalvrisk"] = np.log(df_train["Sjalvrisk"])
df_val["log_sjalvrisk"] = np.log(df_val["Sjalvrisk"])
df_train["log_forsakringsbelopp"] = np.log(df_train["Forsakringsbelopp"])
df_val["log_forsakringsbelopp"] = np.log(df_val["Forsakringsbelopp"])

print(f"Självrisk — unika värden i träningsdata: {sorted(df_train['Sjalvrisk'].unique())}")
print(f"Antal unika självrisknivåer: {df_train['Sjalvrisk'].nunique()}")
print(f"Antal unika försäkringsbelopp: {df_train['Forsakringsbelopp'].nunique()}")


Självrisk — unika värden i träningsdata: [np.float64(5000.0), np.float64(10000.0), np.float64(20000.0), np.float64(50000.0), np.float64(100000.0), np.float64(250000.0)]
Antal unika självrisknivåer: 6
Antal unika försäkringsbelopp: 13683


In [11]:
# M3-sjalvrisk (kontinuerlig): M2 + log(Sjalvrisk)
m3_sjr = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning "
    "+ log_sjalvrisk",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

print(f"M3-självrisk (log)  AIC: {m3_sjr.aic:,.2f}   Deviance: {m3_sjr.deviance:,.2f}")
print(f"M2 referens         AIC: {m2.aic:,.2f}   Deviance: {m2.deviance:,.2f}")
print(f"\nΔAIC (M3 - M2): {m3_sjr.aic - m2.aic:+.2f}")
print(f"\nKoefficient log(Självrisk): {m3_sjr.params['log_sjalvrisk']:.4f}")
print(f"p-värde:                     {m3_sjr.pvalues['log_sjalvrisk']:.4e}")
print(f"Rate ratio:                  {np.exp(m3_sjr.params['log_sjalvrisk']):.4f}")

M3-självrisk (log)  AIC: 136,256.57   Deviance: 107,916.87
M2 referens         AIC: 136,971.03   Deviance: 108,633.33

ΔAIC (M3 - M2): -714.46

Koefficient log(Självrisk): -0.3336
p-värde:                     7.4155e-153
Rate ratio:                  0.7163


In [12]:
# Valideringsdeviance: M3-självrisk vs M2
val_pred_m3_sjr = m3_sjr.predict(df_val, offset=df_val["log_duration"])
val_dev_m3_sjr = poisson_deviance(y_val, val_pred_m3_sjr.values)
val_dev_m2 = poisson_deviance(y_val, val_pred_m2.values)

print(f"Valideringsdeviance (2024):")
print(f"  M2:              {val_dev_m2:,.2f}")
print(f"  M3-självrisk:    {val_dev_m3_sjr:,.2f}")
print(f"  Δ (M3 - M2):    {val_dev_m3_sjr - val_dev_m2:+.2f}")

Valideringsdeviance (2024):
  M2:              41,002.40
  M3-självrisk:    40,689.72
  Δ (M3 - M2):    -312.68


In [13]:
# M4-forsakringsbelopp: M2 + log(Forsakringsbelopp), utan självrisk
m4_fb = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning "
    "+ log_forsakringsbelopp",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

val_pred_m4_fb = m4_fb.predict(df_val, offset=df_val["log_duration"])
val_dev_m4_fb = poisson_deviance(y_val, val_pred_m4_fb.values)

print(f"M4-försäkringsbelopp  AIC: {m4_fb.aic:,.2f}   Deviance: {m4_fb.deviance:,.2f}")
print(f"M2 referens           AIC: {m2.aic:,.2f}   Deviance: {m2.deviance:,.2f}")
print()
print(f"ΔAIC (M4 - M2): {m4_fb.aic - m2.aic:+.2f}")
print(f"ΔValideringsdeviance (M4 - M2): {val_dev_m4_fb - val_dev_m2:+.2f}")
print()
print(f"Koefficient log(Försäkringsbelopp): {m4_fb.params['log_forsakringsbelopp']:.4f}")
print(f"p-värde:                            {m4_fb.pvalues['log_forsakringsbelopp']:.4e}")
print(f"Rate ratio:                         {np.exp(m4_fb.params['log_forsakringsbelopp']):.4f}")
print(f"Rate ratio vid fördubbling:         {np.exp(m4_fb.params['log_forsakringsbelopp'] * np.log(2)):.4f}")


M4-försäkringsbelopp  AIC: 136,888.61   Deviance: 108,548.92
M2 referens           AIC: 136,971.03   Deviance: 108,633.33

ΔAIC (M4 - M2): -82.42
ΔValideringsdeviance (M4 - M2): -31.08

Koefficient log(Försäkringsbelopp): 0.0781
p-värde:                            3.8220e-20
Rate ratio:                         1.0813
Rate ratio vid fördubbling:         1.0557


In [14]:
# M5-alla ekonomiska variabler: M2 + log(Sjalvrisk) + log(Forsakringsbelopp)
m5_all_economic = smf.glm(
    "AntalSkador ~ C(Verksamhet, Treatment(reference='Byggföretag')) "
    "+ C(GeografisktOmrade, Treatment(reference='Landsbyggd')) "
    "+ log_omsattning "
    "+ log_sjalvrisk "
    "+ log_forsakringsbelopp",
    data=df_train,
    family=sm.families.Poisson(),
    offset=df_train["log_duration"],
).fit()

val_pred_m5_all_economic = m5_all_economic.predict(df_val, offset=df_val["log_duration"])
val_dev_m5_all_economic = poisson_deviance(y_val, val_pred_m5_all_economic.values)

print(f"M5-alla ekonomiska    AIC: {m5_all_economic.aic:,.2f}   Deviance: {m5_all_economic.deviance:,.2f}")
print(f"M3-självrisk referens AIC: {m3_sjr.aic:,.2f}   Deviance: {m3_sjr.deviance:,.2f}")
print(f"M4-försäkringsbelopp  AIC: {m4_fb.aic:,.2f}   Deviance: {m4_fb.deviance:,.2f}")
print()
print(f"ΔAIC (M5 - M3): {m5_all_economic.aic - m3_sjr.aic:+.2f}")
print(f"ΔAIC (M5 - M4): {m5_all_economic.aic - m4_fb.aic:+.2f}")
print(f"ΔValideringsdeviance (M5 - M3): {val_dev_m5_all_economic - val_dev_m3_sjr:+.2f}")
print(f"ΔValideringsdeviance (M5 - M4): {val_dev_m5_all_economic - val_dev_m4_fb:+.2f}")
print()
for param, label in [
    ("log_sjalvrisk", "log(Självrisk)"),
    ("log_forsakringsbelopp", "log(Försäkringsbelopp)"),
]:
    coef = m5_all_economic.params[param]
    print(f"{label}: coef={coef:.4f}, p={m5_all_economic.pvalues[param]:.4e}, "
          f"RR={np.exp(coef):.4f}, RR fördubbling={np.exp(coef * np.log(2)):.4f}")


M5-alla ekonomiska    AIC: 136,172.34   Deviance: 107,830.65
M3-självrisk referens AIC: 136,256.57   Deviance: 107,916.87
M4-försäkringsbelopp  AIC: 136,888.61   Deviance: 108,548.92

ΔAIC (M5 - M3): -84.22
ΔAIC (M5 - M4): -716.27
ΔValideringsdeviance (M5 - M3): -31.49
ΔValideringsdeviance (M5 - M4): -313.09

log(Självrisk): coef=-0.3340, p=3.1386e-153, RR=0.7160, RR fördubbling=0.7933
log(Försäkringsbelopp): coef=0.0789, p=1.5318e-20, RR=1.0821, RR fördubbling=1.0562


In [15]:
model_economic_table = pd.DataFrame({
    "Modell": [
        "M2",
        "M3-självrisk",
        "M4-försäkringsbelopp",
        "M5-alla ekonomiska",
    ],
    "AIC": [m2.aic, m3_sjr.aic, m4_fb.aic, m5_all_economic.aic],
    "Train deviance": [m2.deviance, m3_sjr.deviance, m4_fb.deviance, m5_all_economic.deviance],
    "Valideringsdeviance (2024)": [
        val_dev_m2,
        val_dev_m3_sjr,
        val_dev_m4_fb,
        val_dev_m5_all_economic,
    ],
    "Totalt predikterat 2024": [
        val_pred_m2.sum(),
        val_pred_m3_sjr.sum(),
        val_pred_m4_fb.sum(),
        val_pred_m5_all_economic.sum(),
    ],
    "Totalt observerat 2024": [y_val.sum()] * 4,
})

display(model_economic_table.style.format({
    "AIC": "{:,.0f}",
    "Train deviance": "{:,.0f}",
    "Valideringsdeviance (2024)": "{:,.0f}",
    "Totalt predikterat 2024": "{:,.0f}",
    "Totalt observerat 2024": "{:,.0f}",
}))


,Modell,AIC,Train deviance,Valideringsdeviance (2024),Totalt predikterat 2024,Totalt observerat 2024
0,M2,"136,971","108,633","41,002","5,250","5,446"
1,M3-självrisk,"136,257","107,917","40,690","5,248","5,446"
2,M4-försäkringsbelopp,"136,889","108,549","40,971","5,252","5,446"
3,M5-alla ekonomiska,"136,172","107,831","40,658","5,250","5,446"


In [16]:
# Korrelationskontroll mellan ekonomiska variabler
corr_economic = df_train[[
    "log_omsattning",
    "log_sjalvrisk",
    "log_forsakringsbelopp",
]].corr()

display(corr_economic.round(3))


,log_omsattning,log_sjalvrisk,log_forsakringsbelopp
log_omsattning,1.000,0.447,0.573
log_sjalvrisk,0.447,1.000,0.260
log_forsakringsbelopp,0.573,0.260,1.000


### Resultat: ekonomiska variabler

M5 (alla ekonomiska) har lägst AIC och valideringsdeviance, men förbättringen mot M2 är ~0,8 %.

Självrisk och Försäkringsbelopp förbättrar prediktionen något men är svårare att tolka som rena riskfaktorer — de påverkas av försäkringsavtal, kundval och underwriting. Korrelationen med Omsättning (0,45–0,57 på log-skala) innebär överlappande information.

M2 behålls som huvudmodell. M3–M5 rapporteras som robusthetskontroll.

In [17]:
# Extrahera enbart årskoefficienter från M3
year_mask = m3.params.index.str.contains("C(Ar)", regex=False)
year_params = m3.params[year_mask]
year_conf = m3.conf_int().loc[year_mask]

year_table = pd.DataFrame({
    "Koefficient (β)": year_params.values,
    "Rate ratio": np.exp(year_params.values),
    "KI nedre (2.5%)": np.exp(year_conf.iloc[:, 0].values),
    "KI övre (97.5%)": np.exp(year_conf.iloc[:, 1].values),
}, index=year_params.index)

# Renare radnamn
year_table.index = year_table.index.str.extract(r"\[T\.(\d+)\]")[0].values
year_table.index.name = "År (ref: 2021)"

display(year_table.round(4))

,Koefficient (β),Rate ratio,KI nedre (2.5%),KI övre (97.5%)
År (ref: 2021),,,,
2022,-0.0396,0.9612,0.9226,1.0013
2023,0.0077,1.0078,0.9683,1.0488


Årseffekterna visar svag tidsvariation, konsistent med A3. Kan inte extrapoleras → påverkar inte modellvalet.

## 5. Slutsats

**M2 behålls som primär Poisson-GLM.** Enklare, mer transparent, och bygger på variabler som är lättare att tolka som riskfaktorer. M5 förbättrar valideringsdeviance med ~0,8 % men tillför kontraktsrelaterade variabler som inte bör vara centrala i en tariffmodell.

M2-specifikation:
```
AntalSkador ~ C(Verksamhet) + C(GeografisktOmrade) + log(Omsättning) + offset(log(Duration))
```